# 05. Hooks, Saved Tensor Offloading & Autograd Engine Internals

### Historical Lineage & Theoretical Foundations
- **Author / Lineage:** *Linnainmaa (1970); Rumelhart, Hinton & Williams (1986); Paszke et al. (2017, 2019)*.
- **Core Purpose:** Intercepting forward/backward tensors and offloading intermediate activations to break the $\mathcal{O}(B \cdot L \cdot D \cdot N_l)$ VRAM barrier.

### Decision Boundary: When NOT to Use
> [!WARNING]
> - **DO NOT USE module hooks for core arithmetic:** Adding computation inside forward hooks creates hidden side-effects that break `torch.compile` graph tracing.
> - **DO NOT USE `register_backward_hook`:** The legacy backward hook has undefined semantics for multi-tensor view outputs; use `register_full_backward_hook` instead.
> - **DO NOT USE saved tensor hooks when activations are memory-trivial:** Offloading tiny tensors across PCIe/CPU introduces latency overhead exceeding VRAM savings.


### Mathematical Formulation & Asymptotic Complexity

Let $\mathbf{z} = \mathbf{X} \mathbf{W} + \mathbf{b}$ and $\hat{\mathbf{y}} = \text{GELU}(\mathbf{z}) = \mathbf{z} \cdot \Phi(\mathbf{z})$, with $\Phi(z) = \frac{1}{2}[1 + \text{erf}(z / \sqrt{2})]$.

Given scalar loss $\mathcal{L} = \frac{1}{2} \|\hat{\mathbf{y}} - \mathbf{y}_{\text{target}}\|_2^2$:
$$\frac{\partial \mathcal{L}}{\partial \hat{\mathbf{y}}} = \hat{\mathbf{y}} - \mathbf{y}_{\text{target}}$$
$$\frac{\partial \mathcal{L}}{\partial \mathbf{z}} = \frac{\partial \mathcal{L}}{\partial \hat{\mathbf{y}}} \odot \left[ \Phi(\mathbf{z}) + \frac{\mathbf{z}}{\sqrt{2\pi}} e^{-\frac{\mathbf{z}^2}{2}} \right]$$
$$\nabla_{\mathbf{W}} \mathcal{L} = \mathbf{X}^\top \frac{\partial \mathcal{L}}{\partial \mathbf{z}}, \quad \nabla_{\mathbf{b}} \mathcal{L} = \sum_{i} \left(\frac{\partial \mathcal{L}}{\partial \mathbf{z}}\right)_i, \quad \nabla_{\mathbf{X}} \mathcal{L} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}} \mathbf{W}^\top$$

**Asymptotic Complexity:**
- Standard Autograd: Time $\mathcal{O}(B \cdot D_{\text{in}} \cdot D_{\text{out}})$, Activation Memory $\mathcal{O}(B \cdot D_{\text{out}})$.
- Saved Tensor Offload / Packing: Time $\mathcal{O}(B \cdot D_{\text{in}} \cdot D_{\text{out}})$, Device Memory $\mathcal{O}(1)$ or $4\times$ reduction.


In [ ]:
from pathlib import Path

import torch
import torch.nn as nn

import core

# 1. Deterministic Setup
env = core.setup_env(seed=42)

# 2. Ingest Reference Checkpoint
ckpt_path = Path("weights/00_foundations/05_hooks_and_saved_tensor_hooks.safetensors")
weights, test_in, golden_out, meta = core.parity.load_reference_checkpoint(ckpt_path)

w = weights["weight"].to(env.device)
b = weights["bias"].to(env.device)
target = weights["target"].to(env.device)
golden_gw = weights["grad_weight"].to(env.device)
golden_gb = weights["grad_bias"].to(env.device)
golden_gx = weights["grad_input"].to(env.device)

print(f"Device: {env.device} | Input: {test_in.shape} | Targets: {target.shape}")


In [ ]:
# Module and Saved Tensor Hooks Implementation
class HookedLinearLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.weight = nn.Parameter(w.clone().detach())
        self.bias = nn.Parameter(b.clone().detach())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight + self.bias


# Saved Tensors Packing / Offloading Hooks Context Manager
def make_offload_hooks():
    def pack_hook(tensor: torch.Tensor):
        return (tensor.device, tensor.to("cpu", non_blocking=False))

    def unpack_hook(packed):
        orig_device, cpu_tensor = packed
        return cpu_tensor.to(orig_device, non_blocking=False)

    return pack_hook, unpack_hook


model = HookedLinearLayer(3, 2)
pack_fn, unpack_fn = make_offload_hooks()
print("HookedLinearLayer initialized with transparent saved_tensors_hooks offloader.")


In [ ]:
# Quantitative Parity Verification
x_active = test_in.clone().detach().requires_grad_(True)

with torch.autograd.graph.saved_tensors_hooks(pack_fn, unpack_fn):
    pred = model(x_active)
    loss = 0.5 * torch.sum((pred - target) ** 2)
    loss.backward()

# 1. Forward Output Parity
torch.testing.assert_close(pred, golden_out, atol=1e-5, rtol=1e-5)

# 2. Backward Gradient Parity with Saved Tensor Offloading
torch.testing.assert_close(model.weight.grad, golden_gw, atol=1e-5, rtol=1e-5)
torch.testing.assert_close(model.bias.grad, golden_gb, atol=1e-5, rtol=1e-5)
torch.testing.assert_close(x_active.grad, golden_gx, atol=1e-5, rtol=1e-5)

print("All mathematical invariant checks (Forward + Backward Gradient Parity) passed successfully.")


In [ ]:
# Micro-benchmark: Standard Autograd vs Offloaded Saved Tensors vs Dynamic Packing
def run_standard():
    x_s = test_in.clone().detach().requires_grad_(True)
    p = model(x_s)
    l = 0.5 * torch.sum((p - target) ** 2)
    l.backward()

def run_offloaded():
    x_s = test_in.clone().detach().requires_grad_(True)
    with torch.autograd.graph.saved_tensors_hooks(pack_fn, unpack_fn):
        p = model(x_s)
        l = 0.5 * torch.sum((p - target) ** 2)
        l.backward()

track_fns = {
    "Standard PyTorch Autograd": run_standard,
    "Saved Tensors Offload": run_offloaded,
}

bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=test_in.shape[0])
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# [!WARNING]
# PATHOLOGICAL MISUSE & FAILURE MODE: Modifying tensor saved for backward without detaching
# 1. Unsafe forward hook accumulating graph references (Memory Leak)
leaky_history = []
def leaky_forward_hook(module, args, output):
    leaky_history.append(output)  # BUG: Appending un-detached tensor retains entire computation graph!

hook_handle = model.register_forward_hook(leaky_forward_hook)
for _ in range(5):
    _ = model(test_in)
hook_handle.remove()

# Assert that graph references are actively retained
assert leaky_history[0].grad_fn is not None, "Leaky hook retained active grad_fn graph pointer!"
print(f"[FAILURE DEMONSTRATED] Leaky hook retained {len(leaky_history)} un-detached graph nodes in memory!")

# 2. CORRECT PATTERN: Always detach or extract scalar values inside hooks
safe_history = []
def safe_forward_hook(module, args, output):
    safe_history.append(output.detach().clone())  # CORRECT: detached memory snapshot

hook_handle_safe = model.register_forward_hook(safe_forward_hook)
for _ in range(5):
    _ = model(test_in)
hook_handle_safe.remove()

assert safe_history[0].grad_fn is None, "Safe hook must detach from computation graph!"
print(f"[CORRECT] Safe hook successfully decoupled from autograd graph: grad_fn={safe_history[0].grad_fn}")


## Architectural Topology, Key Takeaways & Curriculum Navigation

### Autograd Engine Hook & Saved Tensors Topology
<svg width="650" height="170" xmlns="http://www.w3.org/2000/svg" style="background:#0f172a; border-radius:8px; font-family:system-ui, sans-serif;">
  <defs>
    <marker id="arrow-hook" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#38bdf8"/>
    </marker>
    <marker id="arrow-save" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#f59e0b"/>
    </marker>
  </defs>
  <text x="30" y="25" fill="#94a3b8" font-size="12" font-weight="bold">AUTOGRAD ENGINE INTERNALS &amp; SAVED TENSOR HOOKS</text>
  <!-- Forward Node -->
  <rect x="30" y="50" width="140" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="100" y="72" fill="#38bdf8" font-size="11" text-anchor="middle">Forward: Z = XW + b</text>
  <line x1="170" y1="67" x2="250" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url="#arrow-hook""/>
  <!-- Non-linearity -->
  <rect x="250" y="50" width="140" height="35" rx="6" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="320" y="72" fill="#10b981" font-size="11" text-anchor="middle">Activation: Y = GELU(Z)</text>
  <!-- Intercept & Pack -->
  <line x1="320" y1="85" x2="320" y2="115" stroke="#f59e0b" stroke-width="2" marker-end="url="#arrow-save""/>
  <rect x="240" y="115" width="160" height="35" rx="6" fill="#0f172a" stroke="#f59e0b" stroke-width="1.5" stroke-dasharray="4 4"/>
  <text x="320" y="137" fill="#f59e0b" font-size="11" text-anchor="middle">pack_hook: INT8 / Host RAM</text>
  <!-- Backward Reclaim -->
  <line x1="400" y1="132" x2="480" y2="132" stroke="#f59e0b" stroke-width="2" marker-end="url="#arrow-save""/>
  <rect x="480" y="115" width="140" height="35" rx="6" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="550" y="137" fill="#10b981" font-size="11" text-anchor="middle">unpack_hook: VJP</text>
</svg>

### Cross-Domain Generalisation Matrix
| Scientific Domain | Manifestation of Hooks & Activation Offloading |
| :--- | :--- |
| **Large-Scale Foundation Models** | Offloading activation checkpoints across PCIe/NVLink eliminates VRAM exhaustion during long-context fine-tuning. |
| **AI Interpretability & Probing** | Forward hooks extract intermediate representations across layers without altering network code. |
| **Differential Privacy (DP-SGD)** | Backward hooks calculate per-layer gradient Frobenius norms to calibrate dynamic clipping thresholds. |

### Key Takeaways
1. **Activation Memory Scaling:** In deep networks, $\mathcal{M}_{\text{act}}$ exceeds parameter footprint; `saved_tensors_hooks` provides transparent activation compression and host offloading.
2. **Full Backward Hook Safety:** Always use `register_full_backward_hook` over legacy backward hooks to preserve correct view mechanics.
3. **Detachment Invariant:** Never retain un-detached tensor references inside hooks to avoid permanent autograd memory leaks.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_hooks_and_saved_tensor_hooks.md](../../markdown/00_foundations/05_hooks_and_saved_tensor_hooks.md)
- **PyTorch 2.x Production Module:** [05_hooks_and_saved_tensor_hooks.py](../../pytorch2/00_foundations/05_hooks_and_saved_tensor_hooks.py)
- **Functional JAX Track:** [05_hooks_and_saved_tensor_hooks.py](../../jax/00_foundations/05_hooks_and_saved_tensor_hooks.py)
- **Native Rust Candle Track:** [05_hooks_and_saved_tensor_hooks.rs](../../candle/00_foundations/05_hooks_and_saved_tensor_hooks.rs)
- **Hardware CPU Kernel:** [05_hooks_and_saved_tensor_hooks_cpu.py](../../kernels/00_foundations/05_hooks_and_saved_tensor_hooks_cpu.py)
- **Golden Reference Checkpoint:** [05_hooks_and_saved_tensor_hooks.safetensors](../../weights/00_foundations/05_hooks_and_saved_tensor_hooks.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. Forward AD & Functional Transforms](04_forward_ad_and_functional_transforms_torch_func.ipynb) | [Master Index: README.md](../../README.md) | [Next: 06. Module Internals, Parametrizations & Meta Devices](06_nn_module_internals_parametrizations_and_meta_device.ipynb) &rarr;
